In [18]:
import pandas as pd
from sklearn.datasets import load_wine

wine_data = load_wine()

X = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)

y = wine_data.target

print("First 5 rows of the Wine dataset features:")
display(X.head())

print("\nDataset shape (Rows, Columns):", X.shape)
print("\nFeature names:\n", list(wine_data.feature_names))
print("\nTarget labels:", set(y))
print("Target names:", list(wine_data.target_names))

print("\nClass distribution (Sample count per class):")
print(pd.Series(y).value_counts().sort_index())

First 5 rows of the Wine dataset features:


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0



Dataset shape (Rows, Columns): (178, 13)

Feature names:
 ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']

Target labels: {np.int64(0), np.int64(1), np.int64(2)}
Target names: [np.str_('class_0'), np.str_('class_1'), np.str_('class_2')]

Class distribution (Sample count per class):
0    59
1    71
2    48
Name: count, dtype: int64


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

print("Missing Values And Duplicates")
print("Missing values per column:\n", X.isnull().sum())
print("\nTotal missing values in dataset:", X.isnull().sum().sum())
print("Number of duplicate rows:", X.duplicated().sum())
print("\nData Types")
print(X.dtypes)
print(f"\nNumerical features count: {len(X.select_dtypes(include=[np.number]).columns)}")
print(f"Categorical features count: {len(X.select_dtypes(include=['object', 'category']).columns)}")
print("\nDescriptive Statistics")
stats = X.describe(percentiles=[0.25, 0.5, 0.75]).T
stats["median"] = X.median()
stats_ordered = stats[
    ["mean", "median", "min", "max", "std", "25%", "75%"]
].rename(columns={"25%": "Q1", "75%": "Q3"})
display(stats_ordered)
plt.figure(figsize=(12, 10))
corr_matrix = X.corr()
sns.heatmap(corr_matrix, cmap="coolwarm", annot=False, fmt=".2f")
plt.title("Pearson Feature Correlation Heatmap")
plt.tight_layout()
plt.show()
corr_unstack = corr_matrix.abs().unstack()
corr_unstack = corr_unstack[corr_unstack < 1.0]  
unique_pairs = (
    corr_unstack.reset_index()
    .drop_duplicates(subset=[0])
    .sort_values(by=0, ascending=False))
unique_pairs.columns = ["Feature 1", "Feature 2", "Absolute Correlation"]
print("\nTop 3 Strongest Unique Feature Pairs:")
display(unique_pairs.head(3))

plt.figure(figsize=(15, 8))
X.boxplot(rot=90)
plt.title("Boxplots of Features")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

print("Splitting The Datasets And Verifying")
SEEDS = [42, 52, 62]
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

X_train_dict = {}
X_val_dict = {}
y_train_dict = {}
y_val_dict = {}
split_summary = []

for seed in SEEDS:
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.125, stratify=y_temp, random_state=seed)
    X_train_dict[seed] = X_train
    X_val_dict[seed] = X_val
    y_train_dict[seed] = y_train
    y_val_dict[seed] = y_val
    
    total_samples = len(X)
    split_summary.append({
        "Seed": seed,
        "Subset": "Train (~70%)",
        "Sample Count": len(X_train),
        "Percentage": f"{(len(X_train)/total_samples)*100:.1f}%",
        "Class Distribution": dict(pd.Series(y_train).value_counts().sort_index())})
    split_summary.append({
        "Seed": seed,
        "Subset": "Validation (~10%)",
        "Sample Count": len(X_val),
        "Percentage": f"{(len(X_val)/total_samples)*100:.1f}%",
        "Class Distribution": dict(pd.Series(y_val).value_counts().sort_index())})
    split_summary.append({
        "Seed": seed,
        "Subset": "Fixed Test (20%)",
        "Sample Count": len(X_test),
        "Percentage": f"{(len(X_test)/total_samples)*100:.1f}%",
        "Class Distribution": dict(pd.Series(y_test).value_counts().sort_index())})

summary_df = pd.DataFrame(split_summary)
display(summary_df)
print("\nVerification Checklist:")
print(f"Total dataset size: {len(X)}")
print(f"Train + Val + Test sizes for Seed 42: {len(X_train_dict[42])} + {len(X_val_dict[42])} + {len(X_test)} = {len(X_train_dict[42]) + len(X_val_dict[42]) + len(X_test)}")
print("Are Test indices identical across all seeds? Yes.")

In [19]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

print("Data Preprocessing and Scaling")
print("No missing data for these datasets so no imputation required.")

X_train_scaled_dict = {}
X_val_scaled_dict = {}
X_test_scaled = None

for s in SEEDS:
    scaler = StandardScaler()
    
    current_train = X_train_dict[s]
    current_val = X_val_dict[s]
    scaler.fit(current_train)
    
    train_scaled_array = scaler.transform(current_train)
    val_scaled_array = scaler.transform(current_val)
    X_train_scaled_dict[s] = pd.DataFrame(train_scaled_array, columns=X.columns, index=current_train.index)
    X_val_scaled_dict[s] = pd.DataFrame(val_scaled_array, columns=X.columns, index=current_val.index)
    
    if s == 42:
        test_scaled_array = scaler.transform(X_test)
        X_test_scaled = pd.DataFrame(test_scaled_array, columns=X.columns, index=X_test.index)
        
        print("\n Seed 42 Train Data before Scaling (First 5 Rows)")
        display(current_train.head())
        print("\n Seed 42 Train Data after Scaling (First 5 Rows)")
        display(X_train_scaled_dict[s].head())
        print("\nInterpretation:")
        print("I picked StandardScaler. As we can see, the values changed a lot.")
        print("Now the mean is close to 0 and std is 1.")
        print("This is very important because big numbers (like proline) can mess up algorithms like KNN.")

Data Preprocessing and Scaling
No missing data for these datasets so no imputation required.

 Seed 42 Train Data before Scaling (First 5 Rows)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
163,12.96,3.45,2.35,18.5,106.0,1.39,0.70,0.40,0.94,5.28,0.68,1.75,675.0
148,13.32,3.24,2.38,21.5,92.0,1.93,0.76,0.45,1.25,8.42,0.55,1.62,650.0
116,11.82,1.47,1.99,20.8,86.0,1.98,1.60,0.30,1.53,1.95,0.95,3.33,495.0
122,12.42,4.43,2.73,26.5,102.0,2.20,2.13,0.43,1.71,2.08,0.92,3.12,365.0
115,11.03,1.51,2.20,21.5,85.0,2.46,2.17,0.52,2.01,1.90,1.71,2.87,407.0



 Seed 42 Train Data after Scaling (First 5 Rows)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
163,-0.017514,1.089178,-0.040117,-0.308405,0.433224,-1.342505,-1.322275,0.352668,-1.117639,0.125837,-1.226754,-1.223646,-0.209375
148,0.439706,0.891779,0.072941,0.609409,-0.505699,-0.490993,-1.259892,0.769946,-0.571881,1.512255,-1.811372,-1.414652,-0.291400
116,-1.465379,-0.772006,-1.396814,0.395252,-0.908094,-0.412149,-0.386537,-0.481889,-0.078939,-1.344472,-0.012548,1.097809,-0.799954
122,-0.703345,2.010370,1.391951,2.139100,0.164961,-0.065237,0.164509,0.603035,0.237953,-1.287073,-0.147460,0.789262,-1.226483
115,-2.468723,-0.734407,-0.605407,0.609409,-0.975160,0.344750,0.206097,1.354136,0.766105,-1.366549,3.405216,0.421943,-1.088682



Interpretation:
I picked StandardScaler. As we can see, the values changed a lot.
Now the mean is close to 0 and std is 1.
This is very important because big numbers (like proline) can mess up algorithms like KNN.


In [ ]:
actual_labels = [0, 1, 1, 0, 0, 1, 1, 0, 1, 1]
predictions = [0.14, 0.82, 1.23, 0.71, 0.38, 0.67, 1.29, 0.56, 0.91, 2.11]
tpr_list = []
fpr_list = []
thresholds = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0,
              1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2]

for threshold in thresholds:
    tp = 0
    tn = 0
    fp = 0
    fn = 0
    threshold_results = []
    for p in predictions:
        if p >= threshold:
            threshold_results.append(1)
        else:
            threshold_results.append(0)     
    for i in range(len(actual_labels)):
        if actual_labels[i] == 1 and threshold_results[i] == 1:
            tp += 1
        if actual_labels[i] == 0 and threshold_results[i] == 0:
            tn += 1
        if actual_labels[i] == 0 and threshold_results[i] == 1:
            fp += 1
        if actual_labels[i] == 1 and threshold_results[i] == 0:
            fn += 1
            
    if (tp + fn) == 0:
        tpr = 0.0
    else:
        tpr = tp / (tp + fn)
        
    if (tn + fp) == 0:
        fpr = 0.0
    else:
        fpr = 1.0 - (tn / (tn + fp))
        
    tpr_list.append(tpr)
    fpr_list.append(fpr)
    
    print(f"Threshold: {threshold:.1f} | TP:{tp} TN:{tn} FP:{fp} FN:{fn} | TPR: {tpr:.2f} FPR: {fpr:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.plot(fpr_list, tpr_list, marker='o', color='blue', label='Model ROC Curve')

plt.plot([0, 1], [0, 1], linestyle='--', color='red', label='Random Guess')

plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')

plt.xlim([-0.05, 1.05])
plt.ylim([-0.05, 1.05])

plt.grid(True)
plt.legend()
plt.show()

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import matplotlib.pyplot as plt

wine_data = load_wine()
X_wine = wine_data.data
y_wine = wine_data.target
binary_mask = y_wine != 2
X_wine = X_wine[binary_mask]
y_wine = y_wine[binary_mask]

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_wine, y_wine, test_size=0.3, random_state=42)

dt_wine = DecisionTreeClassifier(random_state=42)
rf_wine = RandomForestClassifier(random_state=42)
knn_wine = KNeighborsClassifier()
svm_wine = SVC(probability=True, random_state=42)
dt_wine.fit(X_train_w, y_train_w)
rf_wine.fit(X_train_w, y_train_w)
knn_wine.fit(X_train_w, y_train_w)
svm_wine.fit(X_train_w, y_train_w)
wine_models_probs = {
    'Decision Tree': dt_wine.predict_proba(X_test_w)[:, 1],
    'Random Forest': rf_wine.predict_proba(X_test_w)[:, 1],
    'KNN': knn_wine.predict_proba(X_test_w)[:, 1],
    'SVM': svm_wine.predict_proba(X_test_w)[:, 1]}

actual_labels_w = list(y_test_w)
thresholds = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
plt.figure(figsize=(9, 6))

for model_name, predictions in wine_models_probs.items():
    tpr_list = []
    fpr_list = []
    for threshold in thresholds:
        tp = 0
        tn = 0
        fp = 0
        fn = 0
        for i in range(len(actual_labels_w)):
            if predictions[i] >= threshold:
                pred = 1
            else:
                pred = 0   
            if actual_labels_w[i] == 1 and pred == 1:
                tp += 1
            if actual_labels_w[i] == 0 and pred == 0:
                tn += 1
            if actual_labels_w[i] == 0 and pred == 1:
                fp += 1
            if actual_labels_w[i] == 1 and pred == 0:
                fn += 1             
        if (tp + fn) == 0:
            tpr = 0.0
        else:
            tpr = tp / (tp + fn)           
        if (tn + fp) == 0:
            fpr = 0.0
        else:
            fpr = 1.0 - (tn / (tn + fp))
            
        tpr_list.append(tpr)
        fpr_list.append(fpr)
        
    plt.plot(fpr_list, tpr_list, marker='o', label=f'{model_name}')
plt.plot([0, 1], [0, 1], linestyle='--', color='red', label='Random Guess')
plt.title('Wine Dataset - Model Comparison ROC Curves')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.xlim([-0.05, 1.05])
plt.ylim([-0.05, 1.05])
plt.grid(True)
plt.legend()
plt.show()